# LEXAI — Leukemia Detection Training

Full training pipeline for the LEXAI 4-class (Normal / ALL / AML / CML) multi-backbone ensemble.

**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

### Architecture
- **CNN Ensemble:** EfficientNet-B0 + ResNet50 + DenseNet121 + ViT-B/16 with learnable fusion weights
- **GNN Pathway:** GCN → GAT → GraphSAGE (optional, for full-field images)
- **3-Stage Training:** Freeze → Fine-tune → Temperature calibration
- **Mixed Precision (AMP):** ~2x faster, ~50% less VRAM

## 0. Check GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 1. Mount Google Drive & Upload Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/LEXAI'
!mkdir -p {DRIVE_DIR}/checkpoints
print(f"Checkpoints will be saved to: {DRIVE_DIR}/checkpoints/")

### Upload your LEXAI project

**Option A** — Upload zip (run the cell below, then upload `LEXAI.zip`):
```bash
# On your laptop, create the zip:
cd ~/LEXAI && zip -r LEXAI.zip lexai/ scripts/ api/ requirements.txt -x '*.pyc' '__pycache__/*' '.git/*' 'data/*' 'checkpoints/*'
```

**Option B** — Clone from GitHub (edit the URL below)

In [ ]:
import os

# === OPTION A: Upload zip ===
USE_UPLOAD = True

# === OPTION B: Clone from GitHub ===
# USE_UPLOAD = False
# GITHUB_REPO = "https://github.com/YOUR_USER/LEXAI.git"

PROJECT_DIR = "/content/LEXAI"

if USE_UPLOAD:
    from google.colab import files
    print("Upload your LEXAI.zip file:")
    uploaded = files.upload()
    !rm -rf {PROJECT_DIR}
    !mkdir -p {PROJECT_DIR} && cd {PROJECT_DIR} && unzip -qo /content/*.zip
    # Handle case where zip contains a root folder
    import glob
    if not os.path.exists(f"{PROJECT_DIR}/lexai") and glob.glob(f"{PROJECT_DIR}/*/lexai"):
        subfolder = glob.glob(f"{PROJECT_DIR}/*/lexai")[0].replace("/lexai", "")
        !mv {subfolder}/* {PROJECT_DIR}/
else:
    !rm -rf {PROJECT_DIR}
    !git clone {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!ls -la
print(f"\nWorking directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
!pip install -q efficientnet-pytorch torch-geometric scikit-learn opencv-python-headless matplotlib
!pip install -q torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html 2>/dev/null || echo "PyG sparse extensions skipped (will use dense fallback)"
!pip install -q kaggle

# Verify
import torch, torchvision, cv2, sklearn
print(f"torch={torch.__version__}, torchvision={torchvision.__version__}")
print(f"opencv={cv2.__version__}, sklearn={sklearn.__version__}")
try:
    import torch_geometric
    print(f"torch_geometric={torch_geometric.__version__}")
except ImportError:
    print("torch_geometric not available — GNN pathway will be disabled")

## 3. Download Datasets

You need Kaggle API credentials. Get them from https://www.kaggle.com/settings → Create New Token.

In [ ]:
import os, json
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    # Option 1: Upload kaggle.json
    print("Upload your kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()
    kaggle_dir.mkdir(exist_ok=True)
    for name, content in uploaded.items():
        kaggle_json.write_bytes(content)
    os.chmod(str(kaggle_json), 0o600)

print(f"Kaggle credentials: {kaggle_json}")
creds = json.loads(kaggle_json.read_text())
print(f"Username: {creds['username']}")

In [ ]:
# Download all datasets (C-NMC, AML-TCIA, Blood Cell Cancer, ALL-IDB, CML)
# This downloads ~5-10 GB and takes 5-15 minutes
!python scripts/download_dataset.py --all --output_dir data

## 4. Prepare & Balance Data

In [ ]:
# Analyze class distribution
!python scripts/prepare_data.py --data_dir data

In [ ]:
# Balance classes (hybrid strategy: undersample majority, oversample minority)
# This generates train/val/test CSV manifests in data/manifests/
# Adjust --target_per_class if you want a specific count per class
!python scripts/prepare_data.py --data_dir data --balance --strategy hybrid

In [ ]:
# Check that manifests were created
!echo "=== Manifests ===" && ls -la data/manifests/
!echo "=== Train manifest (first 5 lines) ===" && head -5 data/manifests/train.csv

## 5. Configure Training

Adjust hyperparameters below. Defaults are tuned for Colab T4 (16 GB VRAM).

In [ ]:
# ====== TRAINING CONFIG ======
DATA_DIR       = "data"                      # raw data directory (images live here)
MANIFEST       = "data/manifests/train.csv"  # balanced manifest from step 4
OUTPUT_DIR     = DRIVE_DIR + "/checkpoints"  # saves to Google Drive
EPOCHS         = 40
BATCH_SIZE     = 16                  # T4 handles 16 with AMP; use 8 if OOM
LR             = 3e-4               # Stage 1 learning rate
FINETUNE_LR    = 5e-5               # Stage 2 learning rate
NUM_WORKERS    = 2                   # Colab has 2 CPU cores
USE_VIT        = True                # ViT-B/16 backbone (biggest model)
USE_GNN        = False               # Disable GNN for single-cell datasets
USE_STAIN_NORM = False               # Macenko stain normalization

# Build command
cmd = f"""python scripts/train.py \
  --data_dir {DATA_DIR} \
  --manifest {MANIFEST} \
  --output_dir {OUTPUT_DIR} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --lr {LR} \
  --finetune_lr {FINETUNE_LR} \
  --num_workers {NUM_WORKERS} \
  --device cuda"""

if not USE_VIT:
    cmd += " --no_vit"
if not USE_GNN:
    cmd += " --no_gnn"
if USE_STAIN_NORM:
    cmd += " --stain_norm"

print("Training command:")
print(cmd)

## 6. Train

In [ ]:
# Run training — this takes 30-90 minutes on T4 depending on dataset size
# Checkpoints auto-save to Google Drive so you won't lose progress
!{cmd}

## 7. Evaluate & Visualize

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load the best checkpoint to see final metrics
ckpt_path = f"{OUTPUT_DIR}/best_model.pth"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    metrics = ckpt.get('metrics', {})
    print("=" * 50)
    print("BEST MODEL METRICS")
    print("=" * 50)
    print(f"  Epoch:     {ckpt.get('epoch', '?')}")
    print(f"  Accuracy:  {metrics.get('accuracy', 0):.4f}")
    print(f"  ECE:       {metrics.get('ece', 0):.4f}")
    print(f"  Precision: {metrics.get('precision', 0):.4f}")
    print(f"  Recall:    {metrics.get('recall', 0):.4f}")
    print(f"  F1:        {metrics.get('f1', 0):.4f}")

calib_path = f"{OUTPUT_DIR}/calibrated_model.pth"
if os.path.exists(calib_path):
    calib = torch.load(calib_path, map_location='cpu', weights_only=False)
    calib_metrics = calib.get('metrics', {})
    print(f"\n  Calibrated ECE: {calib_metrics.get('ece', 0):.4f} (target < 0.05)")

In [ ]:
# Quick validation pass on test set with calibrated model
import sys
sys.path.insert(0, '/content/LEXAI')

from lexai.config import LEXAIConfig
from lexai.models.lexai_model import LEXAIModel
from lexai.data.dataset import create_data_loaders, compute_class_weights
from lexai.training.trainer import Trainer

config = LEXAIConfig()
config.cnn.use_vit = USE_VIT
config.gnn.enabled = USE_GNN

model = LEXAIModel(config)
device = torch.device('cuda')
trainer = Trainer(model, config, device=device, output_dir=OUTPUT_DIR)

# Load calibrated model
if os.path.exists(calib_path):
    trainer.load_checkpoint(calib_path)
elif os.path.exists(ckpt_path):
    trainer.load_checkpoint(ckpt_path)

_, _, test_loader = create_data_loaders(
    data_dir=DATA_DIR, config=config.data,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)

test_loss, test_metrics = trainer.validate(test_loader, calibrate=True)
print("\n" + "=" * 50)
print("TEST SET RESULTS (Calibrated)")
print("=" * 50)
for k, v in sorted(test_metrics.items()):
    if isinstance(v, float):
        print(f"  {k:20s}: {v:.4f}")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        preds = model(images, graph_data=None, calibrate=True)
        all_preds.extend(preds['predicted_class'].cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=config.data.class_names)
disp.plot(ax=ax, cmap='RdPu', values_format='d')
ax.set_title('LEXAI — Test Set Confusion Matrix')
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/confusion_matrix.png", dpi=150)
plt.show()
print(f"Saved to {DRIVE_DIR}/confusion_matrix.png")

## 8. Copy Checkpoint to Drive

Checkpoints are already saved to Drive during training.
Verify they're there:

In [ ]:
import os
print(f"\nCheckpoints in {DRIVE_DIR}/checkpoints/:")
for f in sorted(os.listdir(f"{DRIVE_DIR}/checkpoints")):
    size = os.path.getsize(f"{DRIVE_DIR}/checkpoints/{f}") / 1024**2
    print(f"  {f:30s}  {size:.1f} MB")

print(f"\nTo use on your laptop:")
print(f"  1. Download calibrated_model.pth from Google Drive")
print(f"  2. Place it at: LEXAI/checkpoints/calibrated_model.pth")
print(f"  3. Run: python -m api.server")

## 9. Resume Training (if session disconnected)

If your Colab session disconnected, re-run cells 0-2, then run this cell to resume:

In [ ]:
# Uncomment and run to resume from last checkpoint
# !python scripts/train.py \
#   --data_dir {DATA_DIR} \
#   --output_dir {OUTPUT_DIR} \
#   --epochs {EPOCHS} \
#   --batch_size {BATCH_SIZE} \
#   --lr {LR} \
#   --finetune_lr {FINETUNE_LR} \
#   --num_workers {NUM_WORKERS} \
#   --device cuda \
#   --resume {OUTPUT_DIR}/best_model.pth